# Phase Loop: Batch Phase Search over Multiple .xy Files
Runs the DARA tree-search pipeline from `phase_search copy.ipynb` over every `.xy` file
found in `all_xy_files/`. CIFs are downloaded once and reused for all patterns.

In [1]:
# %pip install ipywidgets nbformat

In [2]:
import os
import re
import time
import tempfile
import requests
from pathlib import Path

from dara import search_phases
from dara.cif import Cif
from dara.structure_db import CODDatabase
from pymatgen.io.cif import CifParser

# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR       = Path("all_xy_files")   # folder containing .xy files to process
CIFS_DIR       = Path("cifs")           # shared CIF cache (reused across patterns)
CHEMICAL_SYSTEM = "Li-Ni-Mn-Co-O-H"
WAVELENGTH      = "Cu"
INSTRUMENT      = "Aeris-fds-Pixcel1d-Medipix3"
EXPRESS_MODE    = False
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR.mkdir(exist_ok=True)
CIFS_DIR.mkdir(exist_ok=True)
print(f"Looking for .xy files in: {DATA_DIR.resolve()}")

Looking for .xy files in: D:\AMIT_WORK\projects\xrd_pipeline\phase search\all_xy_files


## Step 1: Helpers — header stripper & CIF lattice reader

In [3]:
def strip_xy_header(path: Path) -> Path:
    """Return path to a header-stripped copy of the .xy file.

    COMMANDER-exported .xy files have metadata lines at the top that
    DARA cannot parse. Strips non-numeric lines and writes a *_clean.xy
    sibling file. Idempotent: skips re-creation if clean file already exists
    and is newer than the source.
    """
    clean_path = path.parent / (path.stem + "_clean.xy")
    if clean_path.exists() and clean_path.stat().st_mtime >= path.stat().st_mtime:
        return clean_path
    lines = path.read_text(encoding="utf-8").splitlines()
    data_lines = [l for l in lines if re.match(r"^\s*[\d.\-]", l)]
    clean_path.write_text("\n".join(data_lines), encoding="utf-8")
    return clean_path


def cif_stem_to_bgmn_key(stem: str) -> str:
    """Map a CIF filename stem to the BGMN phase key.

    E.g. 'LiNiO2_166_(cod_1533770)-13' -> 'LiNiO2166cod153377013'
    """
    return re.sub(r"[_()\-]", "", stem)


def get_literature_params_nm(cif_path: Path):
    """Return (a, b, c) in nm from a CIF file. Returns (None, None, None) on failure."""
    try:
        parser = CifParser(str(cif_path), occupancy_tolerance=10.0)
        structure = parser.parse_structures(primitive=False)[0]
        lat = structure.lattice
        return lat.a / 10, lat.b / 10, lat.c / 10
    except Exception as exc:
        print(f"    [CIF read error] {cif_path.name}: {exc}")
        return None, None, None

## Step 2: Prepare reference CIFs (downloaded once, cached to `cifs/`)

In [4]:
# Windows fix: COD download retry with exponential back-off
_COD_SESSION = requests.Session()
_COD_SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (research use; contact: your@email.com)"
})
COD_URL = "https://www.crystallography.net/cod/{cod_id}.cif"

def _download_cod_windows_fix(cod_id: str) -> Cif:
    max_retries = 5
    for attempt in range(max_retries):
        try:
            time.sleep(0.5 + attempt * 1.5)
            url = COD_URL.format(cod_id=cod_id)
            response = _COD_SESSION.get(url, timeout=30)
            response.raise_for_status()

            # Windows fix: NamedTemporaryFile keeps the handle open and blocks
            # unlink; use mkstemp for explicit fd control instead.
            fd, temp_path = tempfile.mkstemp(suffix=".cif")
            try:
                os.write(fd, response.content)
                os.close(fd)   # release handle BEFORE Cif.from_file opens it
                fd = -1        # mark as closed so finally-block skips it
                cif = Cif.from_file(temp_path)
            finally:
                if fd != -1:   # close fd if an error occurred before os.close()
                    try:
                        os.close(fd)
                    except OSError:
                        pass
                for _ in range(5):   # retry unlink: Windows may hold the lock briefly
                    try:
                        Path(temp_path).unlink(missing_ok=True)
                        break
                    except PermissionError:
                        time.sleep(0.2)

            cif.filename = str(cod_id)
            return cif
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  [{cod_id}] attempt {attempt+1} failed ({e}), retrying in {wait}s…")
                time.sleep(wait)
            else:
                print(f"  [{cod_id}] all {max_retries} attempts failed: {e}")
                raise

CODDatabase._download_cod = staticmethod(_download_cod_windows_fix)

cod_database = CODDatabase()
all_icsd_ids = cod_database.get_cifs_by_chemsys(CHEMICAL_SYSTEM, dest_dir=str(CIFS_DIR))
print(f"CIF cache ready: {len(list(CIFS_DIR.glob('*.cif')))} files in {CIFS_DIR}")

2026-10-07 16:53:10,432 WARNING dara.structure_db Local copy of database not found. Attempting to download structures...


  [1514116] attempt 1 failed (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))), retrying in 1s…
  [1514239] attempt 1 failed (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))), retrying in 1s…
  [1514110] attempt 1 failed (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))), retrying in 1s…
  [1514232] attempt 1 failed (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))), retrying in 1s…
  [1541154] attempt 1 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 1s…
  [1533885] attempt 1 failed (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))), retrying in 1s…
  [1514102] attempt 1 failed (('Connection aborted.', RemoteDisconnected('Remote end closed connection without

2026-10-07 16:53:48,933 INFO dara.structure_db Saving downloaded CIFs to dara_downloaded_cifs
2026-10-07 16:53:50,401 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,412 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,412 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,412 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,412 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,414 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,414 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,414 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,416 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,416 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,417 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,417 INFO dara.structure_db Skipping common gas: O2
2026-10-07 16:53:50,418 INFO dara.structure_db Skippin

In [5]:
# Windows fix: BGMN holds a file lock on .dia/.str temp files briefly after
# the subprocess exits. Python's TemporaryDirectory.__exit__ raises
# PermissionError [WinError 32] when it tries to unlink those files.
# Setting ignore_cleanup_errors=True (Python 3.10+) silences the cleanup
# error; the OS reclaims the files on next reboot / temp sweep.
import tempfile as _tempfile

_OrigTempDir = _tempfile.TemporaryDirectory

class _WindowsTempDir(_OrigTempDir):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("ignore_cleanup_errors", True)
        super().__init__(*args, **kwargs)

_tempfile.TemporaryDirectory = _WindowsTempDir
print("Windows fix applied: TemporaryDirectory cleanup errors will be ignored.")

Windows fix applied: TemporaryDirectory cleanup errors will be ignored.


## Step 3: Initialise Ray and build curated CIF list

In [6]:
import ray

# Windows fix: stale Ray instances from a previous run cause PermissionError.
if ray.is_initialized():
    ray.shutdown()
ray.init(runtime_env={"working_dir": None})

curated_cifs = [f for f in CIFS_DIR.resolve().glob("*.cif")]
print(f"Curated set: {len(curated_cifs)} phases")

2026-10-07 16:54:00,768	INFO worker.py:2024 -- Started a local Ray instance.


Curated set: 311 phases


In [7]:
# Windows fix: patch remote_do_refinement_no_saving so the tempdir cleanup
# fix runs INSIDE each Ray worker process, not just in the notebook kernel.
#
# The PermissionError [WinError 32] on .dia files happens in the worker because
# BGMN briefly holds a file lock after its subprocess exits. The kernel-side
# _WindowsTempDir patch never reaches workers (separate OS processes). This
# wrapper redefines the remote function so the fix is applied in the worker
# before do_refinement_no_saving opens its TemporaryDirectory.

import ray
import dara.search.tree as _dara_tree
from dara.refine import do_refinement_no_saving as _do_refinement_no_saving

@ray.remote(num_cpus=1)
def _patched_remote_refinement(
    pattern_path,
    cif_paths,
    wavelength,
    instrument_profile,
    phase_params,
    refinement_params,
):
    import tempfile, shutil, time
    import dara.refine as _dr

    # Apply the fix locally inside this worker process
    _OrigTD = tempfile.TemporaryDirectory

    class _RobustTempDir(_OrigTD):
        def __init__(self, *args, **kwargs):
            kwargs.setdefault("ignore_cleanup_errors", True)
            super().__init__(*args, **kwargs)

        def cleanup(self):
            for attempt in range(6):
                try:
                    shutil.rmtree(self.name)
                    return
                except PermissionError:
                    if attempt < 5:
                        time.sleep(0.2 * (attempt + 1))
            shutil.rmtree(self.name, ignore_errors=True)

    tempfile.TemporaryDirectory = _RobustTempDir
    _dr.tempfile.TemporaryDirectory = _RobustTempDir

    # Replicate the original remote function's logic exactly
    from subprocess import TimeoutExpired
    from dara.cif2str import CIF2StrError
    import logging
    logger = logging.getLogger("dara.search.tree")

    if len(cif_paths) == 0:
        return None
    try:
        result = _do_refinement_no_saving(
            pattern_path,
            cif_paths,
            wavelength=wavelength,
            instrument_profile=instrument_profile,
            phase_params=phase_params,
            refinement_params=refinement_params,
        )
    except (RuntimeError, TimeoutExpired, CIF2StrError, ValueError) as e:
        logger.debug(f"Refinement failed for {cif_paths}, the reason is {e}")
        return None
    if result.lst_data.rpb == 100:
        logger.debug(f"Refinement failed for {cif_paths}, the reason is RPB = 100.")
        return None
    return result

# Replace DARA's remote function so batch_refinement picks up our wrapper
_dara_tree.remote_do_refinement_no_saving = _patched_remote_refinement
print("Worker-side Windows fix applied: remote_do_refinement_no_saving patched.")

Worker-side Windows fix applied: remote_do_refinement_no_saving patched.


## Step 4: Batch loop — run phase search on every .xy file

In [8]:
xy_files = sorted(DATA_DIR.glob("*.xy"))
# Exclude already-cleaned copies so we don't process them twice
xy_files = [f for f in xy_files if not f.stem.endswith("_clean")]

if not xy_files:
    print(f"No .xy files found in {DATA_DIR.resolve()}. Add your files and re-run.")
else:
    print(f"Found {len(xy_files)} .xy file(s):")
    for f in xy_files:
        print(f"  {f.name}")

Found 1 .xy file(s):
  OLM-CM 0506026_exported.xy


In [9]:
all_results = {}   # pattern_name -> list[SearchResult]

for xy_file in xy_files:
    print(f"\n{'='*70}")
    print(f"Processing: {xy_file.name}")
    print(f"{'='*70}")

    # Strip header (idempotent)
    clean_path = strip_xy_header(xy_file)
    print(f"  Clean file: {clean_path.name}")

    # Ray must be alive; re-init if it died between iterations
    if not ray.is_initialized():
        ray.init(runtime_env={"working_dir": None})

    search_results = search_phases(
        pattern_path=str(clean_path),
        phases=curated_cifs,
        wavelength=WAVELENGTH,
        instrument_profile=INSTRUMENT,
        express_mode=EXPRESS_MODE,
    )

    all_results[xy_file.name] = search_results

    # Quick summary for this file
    for i, sr in enumerate(search_results):
        rwp = sr.refinement_result.lst_data.rwp
        top_phase = list(sr.refinement_result.lst_data.phases_results.keys())[0]
        print(f"  Solution {i}: Rwp={rwp:.2f}%  top phase={top_phase}")
        sr.visualize()

print(f"\nDone. Processed {len(all_results)} file(s).")


Processing: OLM-CM 0506026_exported.xy
  Clean file: OLM-CM 0506026_exported_clean.xy
2026-10-07 16:54:10,137 INFO dara.search.tree rpb_threshold automatically set to 1.00 based on pattern SNR.
2026-10-07 16:54:10,142 INFO dara.search.tree Detecting peaks in the pattern.
2026-10-07 16:54:13,366 INFO dara.search.tree The wmax is automatically adjusted to 59.09.
2026-10-07 16:54:13,367 INFO dara.search.tree The intensity threshold is automatically set to 10.00 % of maximum peak intensity.
2026-10-07 16:54:13,367 INFO dara.search.tree Creating the root node.
2026-10-07 16:54:13,368 INFO dara.search.tree Refining all the phases in the dataset.
2026-10-07 16:59:11,076 INFO dara.search.tree The initial value of eps2 is automatically set to 0.000003_-0.05^0.05.
2026-10-07 16:59:11,084 INFO dara.search.tree Finished refining 311 phases, with 69 phases removed.
(_remote_expand_node pid=18196) 2026-10-07 16:59:12,002 INFO dara.search.tree Expanding node 9fbae2e0-c241-11f1-a77f-e073e7b8e1ac with

(raylet) Stack (most recent call first):
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdown
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 1170 in wrapper
(raylet) Stack (most recent call first):
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "C:\Users\amitr\AppData\Roaming\Python\Python312\site-packages\ray\_private\worker.py", line 2118 in shutdown
(raylet)   File

(_remote_expand_node pid=18196) 2026-10-07 17:00:03,172 INFO dara.search.tree Expanding node 6eae709f-c242-11f1-911a-e073e7b8e1ac with current phases [RefinementPhase(path=WindowsPath('D:/AMIT_WORK/projects/xrd_pipeline/phase search/cifs/Li0.65Ni1.08O2_166_(cod_1535512)-None.cif'), params={'k1': '0.000000_0.0^0.01', 'b1': '0.003387_0.0^0.005'})], Rwp = 12.12
  Solution 0: Rwp=12.12%  top phase=Li0.65Ni1.08O2_166_(cod_1535512)-None

Done. Processed 1 file(s).


In [10]:
sr.visualize()

## Step 5: Summary table — best solution per file

In [11]:
import pandas as pd

rows = []
for fname, results in all_results.items():
    if not results:
        rows.append({"file": fname, "rwp": None, "top_phase": "no result",
                     "a_nm": None, "c_nm": None})
        continue
    best = results[0]  # DARA returns solutions sorted by FOM; take first
    rwp  = best.refinement_result.lst_data.rwp
    phases_results = best.refinement_result.lst_data.phases_results
    top_key = list(phases_results.keys())[0]
    pr  = phases_results[top_key]
    a_nm = pr.a[0] if pr.a else None
    c_nm = pr.c[0] if pr.c else None
    rows.append({
        "file":      fname,
        "rwp":       rwp,
        "top_phase": top_key,
        "a_nm":      round(a_nm, 6) if a_nm else None,
        "c_nm":      round(c_nm, 6) if c_nm else None,
    })

df_summary = pd.DataFrame(rows)
display(df_summary)

,file,rwp,top_phase,a_nm,c_nm
0,OLM-CM 0506026_exported.xy,12.12,Li0.65Ni1.08O2_166_(cod_1535512)-None,0.286674,1.424737


## Step 6: Detailed lattice parameters — refined vs literature

In [12]:
# for fname, results in all_results.items():
#     print(f"\n{'='*70}")
#     print(f"File: {fname}")
#     print(f"{'='*70}")

#     for sol_idx, result in enumerate(results):
#         rwp = result.refinement_result.lst_data.rwp
#         print(f"\n  Solution {sol_idx}  (Rwp = {rwp:.2f} %)")
#         print(f"  {'-'*66}")

#         phases_results = result.refinement_result.lst_data.phases_results
#         best_phases    = result.phases[0]
#         key_to_path    = {cif_stem_to_bgmn_key(rp.path.stem): rp.path for rp in best_phases}

#         for phase_key, pr in phases_results.items():
#             cif_path = key_to_path.get(phase_key)
#             lit_a, lit_b, lit_c = get_literature_params_nm(cif_path) if cif_path else (None, None, None)
#             label = cif_path.name if cif_path else phase_key

#             print(f"\n    Phase: {label}")
#             print(f"    {'Param':<8} {'Refined (nm)':>14} {'± σ':>10} {'Literature (nm)':>16} {'Δ (%)':>8}")
#             print(f"    {'-'*58}")

#             for pname, refined_tuple, lit_val in [("a", pr.a, lit_a), ("b", pr.b, lit_b), ("c", pr.c, lit_c)]:
#                 if refined_tuple is None:
#                     continue
#                 ref_val, ref_err = refined_tuple
#                 lit_str   = f"{lit_val:.6f}" if lit_val is not None else "N/A"
#                 err_str   = f"± {ref_err:.6f}" if ref_err is not None else ""
#                 delta_str = f"{100 * (ref_val - lit_val) / lit_val:+.3f}%" if lit_val else ""
#                 print(f"    {pname:<8} {ref_val:>14.6f} {err_str:>10} {lit_str:>16} {delta_str:>8}")